## Lab 7: Computing the Spectrum

The goal of this lab is to understand how the **DFT** represents the frequency content of a finite signal and how the same analysis can be repeated over time using the **Short-Time Fourier Transform (STFT)**.

We will focus on four steps:

1. **FFT/window size:** analyze one sinusoid with different values of `N`.
2. **Window type:** compare different analysis windows using the same sinusoid.
3. **Reference sound:** apply what we learned in Sections 1 and 2 to analyze the spectrum of the sound used in the previous labs.
4. **Spectrogram:** extend the DFT analysis over time using the STFT.

The emphasis is on understanding the effect of the analysis parameters rather than implementing the DFT or STFT from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import Audio, display
from util import load_audio, plot_spectrum, plot_spectrogram

### 1. Effect of FFT/window size

We start with a sinusoid because we know exactly which frequency is present.

In the `plot_spectrum()` helper used in this course, the FFT size `N` is the same as the number of analyzed samples. Therefore, changing `N` also changes the **analysis-window duration**.

For a sampling rate $f_s$, the DFT-bin spacing is

$$
\Delta f = \frac{f_s}{N}.
$$

A larger `N` means:

- a longer observation of the signal,
- smaller spacing between DFT bins,
- better ability to distinguish nearby frequencies.

We use a **rectangular window** in this section so that only the effect of changing `N` is studied.

In [ ]:
# Sampling parameters
fs = 44100
f0 = 437

# Compare several FFT/window sizes
for N in [512, 2048, 8192]:

    t = np.arange(N) / fs
    x = np.cos(2 * np.pi * f0 * t)

    delta_f = fs / N
    duration_ms = 1000 * N / fs

    print(
        f"N = {N:4d} | window = {duration_ms:6.1f} ms "
        f"| DFT-bin spacing = {delta_f:6.2f} Hz"
    )

    ax = plot_spectrum(x, w=np.ones(N), N=N, sr=fs)
    ax.set_xlim([0, 1200])
    ax.set_title(f'Sinusoid at {f0} Hz — N={N}')
    plt.show()

#### 1.1 Questions

1. At approximately which frequency is the main spectral peak?
   - Answer:

2. Calculate the window duration and the DFT-bin spacing for each value of `N`.
   - Answer:

3. What happens to the width and definition of the spectral peak as `N` increases?
   - Answer:

4. Which value of `N` gives the best frequency resolution? Why?
   - Answer:

5. Why does the 437-Hz sinusoid spread over several frequency bins instead of appearing in only one bin?
   - Answer:

### 2. Effect of the window type

We now keep the FFT/window size fixed and change only the **window shape**.

A finite signal segment has abrupt boundaries. The analysis window controls how strongly these boundaries affect the spectrum.

We will compare:

- **Rectangular window:** narrow mainlobe, but relatively high sidelobes.
- **Hann window:** lower sidelobes, but a wider mainlobe.
- **Hamming window:** also reduces sidelobes and is commonly used for spectral analysis.

Use the same sinusoid and the same value of `N` in all cases.

In [ ]:
fs = 44100
f0 = 437
N = 2048

t = np.arange(N) / fs
x = np.cos(2 * np.pi * f0 * t)

windows = {
    'Rectangular': np.ones(N),
    'Hann': np.hanning(N),
    'Hamming': np.hamming(N),
}

for name, w in windows.items():

    ax = plot_spectrum(x, w=w, N=N, sr=fs)

    ax.set_xlim([0, 1200])
    ax.set_title(f'{name} window — N={N}')
    plt.show()

#### 2.1 Questions

1. Compare the width of the main peak for the three windows.
   - Answer:

2. Compare the amount of spectral energy away from the main peak.
   - Answer:

3. Which window gives the narrowest mainlobe?
   - Answer:

4. Which window reduces spectral leakage the most clearly?
   - Answer:

5. Explain the basic trade-off between **mainlobe width** and **sidelobe level**.
   - Answer:

6. Based on these observations, which window would you normally choose to analyze a harmonic sound? Explain.
   - Answer:

### 3. DFT analysis of the reference sound

Now apply the observations from Sections 1 and 2 to the **reference sound used in the previous labs**.

The objective is to choose an appropriate:

- FFT/window size `N`,
- window type,
- approximately stationary part of the sound,

and then interpret the harmonic structure of the resulting spectrum.

Use the same reference sound as in Lab 6.

In [ ]:
# Point this path to the same reference sound used in the previous labs.
# Example:
# reference_path = Path('./sis1_name/your_file.wav')

reference_path = Path('audio/reference.wav')
reference_audio, fs = load_audio(reference_path)

# Normalize for listening and visualization
x_ref = reference_audio / np.max(np.abs(reference_audio))

print(f"Sampling rate: {fs} Hz")
print(f"Duration: {len(x_ref) / fs:.3f} s")

display(Audio(x_ref, rate=fs))

t_ref = np.arange(len(x_ref)) / fs

plt.figure(figsize=(10, 3))
plt.plot(t_ref, x_ref)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Reference sound')
plt.grid(True, alpha=0.3)
plt.show()

Choose a relatively stable part of the sound and compare a **shorter** and a **longer** analysis window.

Then select the window type that you found most useful in Section 2.

In [ ]:
# Choose an approximately stationary part of your sound
start_time = 0.20   # seconds

# Compare two FFT/window sizes
for N in [1024, 4096]:

    start = int(start_time * fs)
    segment = x_ref[start:start + N]

    if len(segment) < N:
        print(f"Skipping N={N}: segment extends beyond the end of the sound.")
        continue

    w = np.hamming(N)

    ax = plot_spectrum(
        segment,
        w=w,
        N=N,
        sr=fs
    )

    ax.set_xlim([0, min(5000, fs / 2)])
    ax.set_title(
        f'Reference sound — start={start_time:.2f}s, '
        f'N={N}, Hamming window'
    )
    plt.show()

Now choose the `N` and window type that you consider most appropriate and produce your final spectrum.

In [ ]:
# Choose your final analysis parameters
start_time = 0.20
N = 4096

start = int(start_time * fs)
segment = x_ref[start:start + N]

# Change this window if your observations in Section 2 suggest another choice
w = np.hamming(N)

ax = plot_spectrum(
    segment,
    w=w,
    N=N,
    sr=fs
)

ax.set_xlim([0, min(5000, fs / 2)])
ax.set_title('Final DFT analysis of the reference sound')
plt.show()

#### 3.1 Questions

1. Which FFT/window size gives the most useful spectrum for your reference sound? Why?
   - Answer:

2. Which window type did you choose? Why?
   - Answer:

3. Estimate the fundamental frequency $f_0$ from the spectrum.
   - Answer:

4. Identify at least four harmonic peaks and report their approximate frequencies.
   - Answer:

5. How do these peaks relate to the harmonic components analyzed and synthesized in the previous labs?
   - Answer:

6. What information about the sound is lost when we describe it using only this single DFT?
   - Answer:

### 4. Spectrogram: extending the DFT over time

A single DFT describes only one short segment of the sound. Musical sounds change over time, so we need to repeat the spectral analysis at successive time positions.

The **Short-Time Fourier Transform (STFT)** does exactly this:

1. select a short segment,
2. multiply it by an analysis window,
3. compute its DFT,
4. move forward by `H` samples,
5. repeat.

The spectrogram displays the magnitude of these successive spectra:

- horizontal axis: time,
- vertical axis: frequency,
- intensity/color: spectral magnitude.

We will use the course helper:

```python
plot_spectrogram(x, sr, w=None, N=None, H=256)
```

Here `N` controls the FFT/window size and `H` controls the **hop size**, i.e. how far the window moves between successive analyses.

In [ ]:
# Compare a shorter and a longer STFT analysis window

for N, H in [(1024, 256), (4096, 1024)]:

    w = np.hamming(N)

    print(
        f"N={N}, H={H} | "
        f"window={1000*N/fs:.1f} ms | "
        f"hop={1000*H/fs:.1f} ms"
    )

    ax = plot_spectrogram(x_ref, sr=fs, w=w, N=N, H=H)

    ax.set_ylim([0, min(5000, fs / 2)])
    ax.set_title(f'Reference sound spectrogram — N={N}, H={H}')
    plt.show()

#### 4.1 Questions

1. What does one vertical slice of the spectrogram represent?
   - Answer:

2. Which value of `N` makes individual harmonics easier to distinguish?
   - Answer:

3. Which value of `N` makes changes in time easier to locate?
   - Answer:

4. Explain the time-frequency trade-off observed between the two spectrograms.
   - Answer:

5. What is the role of the hop size `H`?
   - Answer:

6. Compare the spectrogram with the single DFT from Section 3. What additional information does the spectrogram provide?
   - Answer:

7. Relate what you see in the spectrogram to the changing harmonic envelopes studied in Lab 6.
   - Answer: